# P&L Optimization and Sales Maximization

This notebook demonstrates how to connect causal inference results to business optimization problems:
- Profit and Loss (P&L) optimization
- Sales maximization
- Multi-objective optimization
- Constrained optimization

## Objective
Use causal inference insights to optimize business metrics while maintaining operational constraints.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize, differential_evolution
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from utils import generate_synthetic_insurance_data, optimize_pricing_simple, calculate_profit_metrics, set_style

# Set plotting style
set_style()

print("Libraries imported successfully!")

## 1. Generate Data and Build Causal Model

Create dataset and build a causal model for optimization.

In [ ]:
# Generate synthetic insurance data
np.random.seed(42)
df = generate_synthetic_insurance_data(n_samples=10000)

print(f"Dataset shape: {df.shape}")
print(f"\nCurrent Business Metrics:")
current_metrics = calculate_profit_metrics(df)
for key, value in current_metrics.items():
    if isinstance(value, (int, float)):
        if 'rate' in key or 'margin' in key:
            print(f"  {key}: {value:.2%}")
        else:
            print(f"  {key}: {value:,.2f}")

# Build causal model for conversion prediction
features = ['age', 'income', 'risk_score', 'previous_claims', 'price']
X = df[features]
y_conversion = df['conversion']
y_profit = df['profit']

# Train models
conversion_model = RandomForestRegressor(n_estimators=100, random_state=42)
profit_model = RandomForestRegressor(n_estimators=100, random_state=42)

conversion_model.fit(X, y_conversion)
profit_model.fit(X, y_profit)

print(f"\nModel Performance:")
print(f"  Conversion model R²: {conversion_model.score(X, y_conversion):.4f}")
print(f"  Profit model R²: {profit_model.score(X, y_profit):.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'conversion_importance': conversion_model.feature_importances_,
    'profit_importance': profit_model.feature_importances_
})

print(f"\nFeature Importance:")
print(feature_importance.sort_values('conversion_importance', ascending=False))

## 2. P&L Optimization Framework

Set up optimization framework for profit maximization.

In [ ]:
class ProfitOptimizer:
    def __init__(self, data, conversion_model, profit_model, features):
        self.data = data
        self.conversion_model = conversion_model
        self.profit_model = profit_model
        self.features = features
        self.base_data = data[features].copy()
        
    def predict_metrics(self, price, customer_segment=None):
        """Predict conversion and profit for given price."""
        if customer_segment is None:
            # Use all customers
            X_pred = self.base_data.copy()
        else:
            # Use specific segment
            X_pred = customer_segment.copy()
        
        X_pred['price'] = price
        
        # Predict conversion and profit
        conversion_pred = self.conversion_model.predict(X_pred)
        profit_pred = self.profit_model.predict(X_pred)
        
        # Calculate aggregate metrics
        avg_conversion = conversion_pred.mean()
        total_customers = len(X_pred)
        expected_conversions = avg_conversion * total_customers
        total_profit = profit_pred.sum()
        total_revenue = expected_conversions * price
        
        return {
            'price': price,
            'conversion_rate': avg_conversion,
            'expected_conversions': expected_conversions,
            'total_profit': total_profit,
            'total_revenue': total_revenue,
            'profit_per_customer': total_profit / total_customers,
            'revenue_per_customer': total_revenue / total_customers
        }
    
    def optimize_profit(self, price_bounds=(200, 2000), method='bounded'):
        """Optimize for maximum profit."""
        def objective(price):
            metrics = self.predict_metrics(price[0])
            return -metrics['total_profit']  # Negative for minimization
        
        if method == 'bounded':
            result = minimize(objective, x0=[800], bounds=[price_bounds], method='L-BFGS-B')
        else:
            result = differential_evolution(objective, bounds=[price_bounds], seed=42)
        
        optimal_price = result.x[0]
        optimal_metrics = self.predict_metrics(optimal_price)
        
        return optimal_price, optimal_metrics, result
    
    def optimize_sales(self, price_bounds=(200, 2000)):
        """Optimize for maximum sales (conversions)."""
        def objective(price):
            metrics = self.predict_metrics(price[0])
            return -metrics['expected_conversions']  # Negative for minimization
        
        result = minimize(objective, x0=[800], bounds=[price_bounds], method='L-BFGS-B')
        optimal_price = result.x[0]
        optimal_metrics = self.predict_metrics(optimal_price)
        
        return optimal_price, optimal_metrics, result
    
    def multi_objective_optimize(self, price_bounds=(200, 2000), profit_weight=0.7, sales_weight=0.3):
        """Multi-objective optimization balancing profit and sales."""
        def objective(price):
            metrics = self.predict_metrics(price[0])
            
            # Normalize metrics for fair comparison
            profit_normalized = metrics['total_profit'] / 1000000  # Scale down
            sales_normalized = metrics['expected_conversions'] / 10000  # Scale down
            
            # Weighted combination (negative for minimization)
            return -(profit_weight * profit_normalized + sales_weight * sales_normalized)
        
        result = minimize(objective, x0=[800], bounds=[price_bounds], method='L-BFGS-B')
        optimal_price = result.x[0]
        optimal_metrics = self.predict_metrics(optimal_price)
        
        return optimal_price, optimal_metrics, result

# Initialize optimizer
optimizer = ProfitOptimizer(df, conversion_model, profit_model, features)

print("P&L Optimization Framework initialized!")
print("Available optimization methods:")
print("  1. Profit maximization")
print("  2. Sales maximization")
print("  3. Multi-objective optimization")
print("  4. Constrained optimization")

## 3. Single Objective Optimization

Optimize for individual business objectives.

In [ ]:
print("SINGLE OBJECTIVE OPTIMIZATION")
print("=" * 50)

# 1. Profit maximization
profit_price, profit_metrics, profit_result = optimizer.optimize_profit()
print(f"1. PROFIT MAXIMIZATION:")
print(f"   Optimal price: ${profit_price:.2f}")
print(f"   Expected total profit: ${profit_metrics['total_profit']:,.2f}")
print(f"   Expected conversion rate: {profit_metrics['conversion_rate']:.2%}")
print(f"   Expected conversions: {profit_metrics['expected_conversions']:,.0f}")
print(f"   Profit per customer: ${profit_metrics['profit_per_customer']:.2f}")

# 2. Sales maximization
sales_price, sales_metrics, sales_result = optimizer.optimize_sales()
print(f"\n2. SALES MAXIMIZATION:")
print(f"   Optimal price: ${sales_price:.2f}")
print(f"   Expected conversions: {sales_metrics['expected_conversions']:,.0f}")
print(f"   Expected conversion rate: {sales_metrics['conversion_rate']:.2%}")
print(f"   Expected total profit: ${sales_metrics['total_profit']:,.2f}")
print(f"   Profit per customer: ${sales_metrics['profit_per_customer']:.2f}")

# 3. Multi-objective optimization
multi_price, multi_metrics, multi_result = optimizer.multi_objective_optimize()
print(f"\n3. MULTI-OBJECTIVE OPTIMIZATION (70% profit, 30% sales):")
print(f"   Optimal price: ${multi_price:.2f}")
print(f"   Expected total profit: ${multi_metrics['total_profit']:,.2f}")
print(f"   Expected conversions: {multi_metrics['expected_conversions']:,.0f}")
print(f"   Expected conversion rate: {multi_metrics['conversion_rate']:.2%}")
print(f"   Profit per customer: ${multi_metrics['profit_per_customer']:.2f}")

# Current performance for comparison
current_price = df['price'].mean()
current_metrics_pred = optimizer.predict_metrics(current_price)
print(f"\nCURRENT PERFORMANCE:")
print(f"   Current price: ${current_price:.2f}")
print(f"   Current total profit: ${current_metrics_pred['total_profit']:,.2f}")
print(f"   Current conversions: {current_metrics_pred['expected_conversions']:,.0f}")
print(f"   Current conversion rate: {current_metrics_pred['conversion_rate']:.2%}")
print(f"   Current profit per customer: ${current_metrics_pred['profit_per_customer']:.2f}")

# Compare improvements
print(f"\nIMPROVEMENT ANALYSIS:")
profit_improvement = (profit_metrics['total_profit'] - current_metrics_pred['total_profit']) / current_metrics_pred['total_profit']
sales_improvement = (sales_metrics['expected_conversions'] - current_metrics_pred['expected_conversions']) / current_metrics_pred['expected_conversions']
multi_profit_improvement = (multi_metrics['total_profit'] - current_metrics_pred['total_profit']) / current_metrics_pred['total_profit']
multi_sales_improvement = (multi_metrics['expected_conversions'] - current_metrics_pred['expected_conversions']) / current_metrics_pred['expected_conversions']

print(f"   Profit optimization: {profit_improvement:.2%} profit improvement")
print(f"   Sales optimization: {sales_improvement:.2%} sales improvement")
print(f"   Multi-objective: {multi_profit_improvement:.2%} profit, {multi_sales_improvement:.2%} sales")

# Visualization
price_range = np.linspace(200, 2000, 100)
profits = []
conversions = []
revenues = []

for price in price_range:
    metrics = optimizer.predict_metrics(price)
    profits.append(metrics['total_profit'])
    conversions.append(metrics['expected_conversions'])
    revenues.append(metrics['total_revenue'])

# Create plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Profit vs Price
axes[0, 0].plot(price_range, profits, 'b-', linewidth=2)
axes[0, 0].axvline(profit_price, color='r', linestyle='--', label=f'Profit Max: ${profit_price:.0f}')
axes[0, 0].axvline(current_price, color='g', linestyle='--', label=f'Current: ${current_price:.0f}')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Total Profit ($)')
axes[0, 0].set_title('Profit Optimization')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Sales vs Price
axes[0, 1].plot(price_range, conversions, 'g-', linewidth=2)
axes[0, 1].axvline(sales_price, color='r', linestyle='--', label=f'Sales Max: ${sales_price:.0f}')
axes[0, 1].axvline(current_price, color='g', linestyle='--', label=f'Current: ${current_price:.0f}')
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Expected Conversions')
axes[0, 1].set_title('Sales Optimization')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Revenue vs Price
axes[1, 0].plot(price_range, revenues, 'purple', linewidth=2)
axes[1, 0].axvline(multi_price, color='r', linestyle='--', label=f'Multi-Obj: ${multi_price:.0f}')
axes[1, 0].axvline(current_price, color='g', linestyle='--', label=f'Current: ${current_price:.0f}')
axes[1, 0].set_xlabel('Price ($)')
axes[1, 0].set_ylabel('Total Revenue ($)')
axes[1, 0].set_title('Revenue Optimization')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Comparison bar chart
strategies = ['Current', 'Profit Max', 'Sales Max', 'Multi-Obj']
profit_values = [current_metrics_pred['total_profit'], profit_metrics['total_profit'], 
                sales_metrics['total_profit'], multi_metrics['total_profit']]
sales_values = [current_metrics_pred['expected_conversions'], profit_metrics['expected_conversions'],
               sales_metrics['expected_conversions'], multi_metrics['expected_conversions']]

x = np.arange(len(strategies))
width = 0.35

axes[1, 1].bar(x - width/2, [p/1000 for p in profit_values], width, label='Profit (000s)', color='lightblue')
axes[1, 1].bar(x + width/2, [s/10 for s in sales_values], width, label='Sales (10s)', color='lightcoral')
axes[1, 1].set_xlabel('Strategy')
axes[1, 1].set_ylabel('Scaled Values')
axes[1, 1].set_title('Strategy Comparison')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(strategies)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Constrained Optimization

Optimize with business constraints and operational limits.

In [ ]:
print("CONSTRAINED OPTIMIZATION")
print("=" * 50)

# Define business constraints
constraints = {
    'min_conversion_rate': 0.35,  # Minimum 35% conversion rate
    'max_price_increase': 0.30,   # Maximum 30% price increase
    'min_profit_margin': 0.25,    # Minimum 25% profit margin
    'min_sales_volume': 3000      # Minimum 3000 conversions
}

def constrained_profit_optimization(price_bounds=(200, 2000)):
    """Optimize profit subject to business constraints."""
    
    def objective(price):
        metrics = optimizer.predict_metrics(price[0])
        return -metrics['total_profit']  # Negative for minimization
    
    def constraint_conversion_rate(price):
        metrics = optimizer.predict_metrics(price[0])
        return metrics['conversion_rate'] - constraints['min_conversion_rate']
    
    def constraint_price_increase(price):
        max_price = current_price * (1 + constraints['max_price_increase'])
        return max_price - price[0]
    
    def constraint_sales_volume(price):
        metrics = optimizer.predict_metrics(price[0])
        return metrics['expected_conversions'] - constraints['min_sales_volume']
    
    def constraint_profit_margin(price):
        metrics = optimizer.predict_metrics(price[0])
        profit_margin = metrics['total_profit'] / metrics['total_revenue'] if metrics['total_revenue'] > 0 else 0
        return profit_margin - constraints['min_profit_margin']
    
    # Set up constraints
    cons = [
        {'type': 'ineq', 'fun': constraint_conversion_rate},
        {'type': 'ineq', 'fun': constraint_price_increase},
        {'type': 'ineq', 'fun': constraint_sales_volume},
        {'type': 'ineq', 'fun': constraint_profit_margin}
    ]
    
    # Optimize
    result = minimize(objective, x0=[current_price], bounds=[price_bounds], 
                     constraints=cons, method='SLSQP')
    
    return result

# Run constrained optimization
constrained_result = constrained_profit_optimization()

if constrained_result.success:
    constrained_price = constrained_result.x[0]
    constrained_metrics = optimizer.predict_metrics(constrained_price)
    
    print(f"CONSTRAINED OPTIMIZATION RESULTS:")
    print(f"   Optimal price: ${constrained_price:.2f}")
    print(f"   Expected total profit: ${constrained_metrics['total_profit']:,.2f}")
    print(f"   Expected conversion rate: {constrained_metrics['conversion_rate']:.2%}")
    print(f"   Expected conversions: {constrained_metrics['expected_conversions']:,.0f}")
    print(f"   Profit margin: {constrained_metrics['total_profit']/constrained_metrics['total_revenue']:.2%}")
    print(f"   Price increase: {(constrained_price - current_price)/current_price:.2%}")
    
    # Check constraints
    print(f"\nCONSTRAINT VERIFICATION:")
    print(f"   Conversion rate: {constrained_metrics['conversion_rate']:.2%} (min: {constraints['min_conversion_rate']:.2%})")
    print(f"   Price increase: {(constrained_price - current_price)/current_price:.2%} (max: {constraints['max_price_increase']:.2%})")
    print(f"   Sales volume: {constrained_metrics['expected_conversions']:,.0f} (min: {constraints['min_sales_volume']:,})")
    print(f"   Profit margin: {constrained_metrics['total_profit']/constrained_metrics['total_revenue']:.2%} (min: {constraints['min_profit_margin']:.2%})")
    
    # Compare with unconstrained optimization
    unconstrained_improvement = (profit_metrics['total_profit'] - current_metrics_pred['total_profit']) / current_metrics_pred['total_profit']
    constrained_improvement = (constrained_metrics['total_profit'] - current_metrics_pred['total_profit']) / current_metrics_pred['total_profit']
    
    print(f"\nCOMPARISON:")
    print(f"   Unconstrained profit improvement: {unconstrained_improvement:.2%}")
    print(f"   Constrained profit improvement: {constrained_improvement:.2%}")
    print(f"   Constraint penalty: {unconstrained_improvement - constrained_improvement:.2%}")
    
else:
    print(f"Optimization failed: {constrained_result.message}")
    constrained_price = current_price
    constrained_metrics = current_metrics_pred

# Sensitivity analysis for constraints
print(f"\nCONSTRAINT SENSITIVITY ANALYSIS:")
print("=" * 40)

# Test different constraint values
sensitivity_results = []

for conversion_min in [0.30, 0.35, 0.40, 0.45]:
    temp_constraints = constraints.copy()
    temp_constraints['min_conversion_rate'] = conversion_min
    
    # Quick test - just check if current solution satisfies
    test_metrics = optimizer.predict_metrics(constrained_price)
    
    sensitivity_results.append({
        'constraint': f'Min conversion {conversion_min:.2%}',
        'feasible': test_metrics['conversion_rate'] >= conversion_min,
        'slack': test_metrics['conversion_rate'] - conversion_min
    })

for result in sensitivity_results:
    status = "✓" if result['feasible'] else "✗"
    print(f"   {status} {result['constraint']}: slack = {result['slack']:.4f}")

# Visualize constraints
price_test_range = np.linspace(200, 1500, 50)
conversion_rates = []
sales_volumes = []
profit_margins = []

for price in price_test_range:
    metrics = optimizer.predict_metrics(price)
    conversion_rates.append(metrics['conversion_rate'])
    sales_volumes.append(metrics['expected_conversions'])
    profit_margins.append(metrics['total_profit'] / metrics['total_revenue'] if metrics['total_revenue'] > 0 else 0)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(price_test_range, conversion_rates, 'b-', linewidth=2)
plt.axhline(y=constraints['min_conversion_rate'], color='r', linestyle='--', label='Min Constraint')
plt.axvline(x=constrained_price, color='g', linestyle='--', label='Optimal Price')
plt.xlabel('Price ($)')
plt.ylabel('Conversion Rate')
plt.title('Conversion Rate Constraint')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(price_test_range, sales_volumes, 'g-', linewidth=2)
plt.axhline(y=constraints['min_sales_volume'], color='r', linestyle='--', label='Min Constraint')
plt.axvline(x=constrained_price, color='g', linestyle='--', label='Optimal Price')
plt.xlabel('Price ($)')
plt.ylabel('Sales Volume')
plt.title('Sales Volume Constraint')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(price_test_range, profit_margins, 'purple', linewidth=2)
plt.axhline(y=constraints['min_profit_margin'], color='r', linestyle='--', label='Min Constraint')
plt.axvline(x=constrained_price, color='g', linestyle='--', label='Optimal Price')
plt.xlabel('Price ($)')
plt.ylabel('Profit Margin')
plt.title('Profit Margin Constraint')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Customer Segmentation Optimization

Optimize pricing for different customer segments.

In [ ]:
print("CUSTOMER SEGMENTATION OPTIMIZATION")
print("=" * 50)

# Define customer segments
df['age_group'] = pd.cut(df['age'], bins=[0, 35, 50, 65, 100], labels=['Young', 'Middle', 'Senior', 'Elderly'])
df['income_group'] = pd.cut(df['income'], bins=3, labels=['Low', 'Medium', 'High'])
df['risk_group'] = pd.cut(df['risk_score'], bins=3, labels=['Low Risk', 'Medium Risk', 'High Risk'])

# Segment-specific optimization
segment_results = {}

# Age group optimization
print("AGE GROUP OPTIMIZATION:")
for age_group in df['age_group'].unique():
    if pd.isna(age_group):
        continue
    
    segment_data = df[df['age_group'] == age_group]
    segment_features = segment_data[features]
    
    # Simple optimization for this segment
    segment_optimizer = ProfitOptimizer(segment_data, conversion_model, profit_model, features)
    
    # Current performance
    current_segment_price = segment_data['price'].mean()
    current_segment_metrics = segment_optimizer.predict_metrics(current_segment_price, segment_features)
    
    # Optimize for this segment
    optimal_price, optimal_metrics, result = segment_optimizer.optimize_profit()
    
    improvement = (optimal_metrics['total_profit'] - current_segment_metrics['total_profit']) / current_segment_metrics['total_profit']
    
    segment_results[f'age_{age_group}'] = {
        'segment_size': len(segment_data),
        'current_price': current_segment_price,
        'optimal_price': optimal_price,
        'current_profit': current_segment_metrics['total_profit'],
        'optimal_profit': optimal_metrics['total_profit'],
        'improvement': improvement,
        'current_conversion': current_segment_metrics['conversion_rate'],
        'optimal_conversion': optimal_metrics['conversion_rate']
    }
    
    print(f"   {age_group}: Current ${current_segment_price:.0f} → Optimal ${optimal_price:.0f} ({improvement:.1%} profit improvement)")

# Region optimization
print(f"\nREGION OPTIMIZATION:")
for region in df['region'].unique():
    segment_data = df[df['region'] == region]
    segment_features = segment_data[features]
    
    segment_optimizer = ProfitOptimizer(segment_data, conversion_model, profit_model, features)
    
    current_segment_price = segment_data['price'].mean()
    current_segment_metrics = segment_optimizer.predict_metrics(current_segment_price, segment_features)
    
    optimal_price, optimal_metrics, result = segment_optimizer.optimize_profit()
    
    improvement = (optimal_metrics['total_profit'] - current_segment_metrics['total_profit']) / current_segment_metrics['total_profit']
    
    segment_results[f'region_{region}'] = {
        'segment_size': len(segment_data),
        'current_price': current_segment_price,
        'optimal_price': optimal_price,
        'current_profit': current_segment_metrics['total_profit'],
        'optimal_profit': optimal_metrics['total_profit'],
        'improvement': improvement,
        'current_conversion': current_segment_metrics['conversion_rate'],
        'optimal_conversion': optimal_metrics['conversion_rate']
    }
    
    print(f"   {region}: Current ${current_segment_price:.0f} → Optimal ${optimal_price:.0f} ({improvement:.1%} profit improvement)")

# Risk group optimization
print(f"\nRISK GROUP OPTIMIZATION:")
for risk_group in df['risk_group'].unique():
    if pd.isna(risk_group):
        continue
        
    segment_data = df[df['risk_group'] == risk_group]
    segment_features = segment_data[features]
    
    segment_optimizer = ProfitOptimizer(segment_data, conversion_model, profit_model, features)
    
    current_segment_price = segment_data['price'].mean()
    current_segment_metrics = segment_optimizer.predict_metrics(current_segment_price, segment_features)
    
    optimal_price, optimal_metrics, result = segment_optimizer.optimize_profit()
    
    improvement = (optimal_metrics['total_profit'] - current_segment_metrics['total_profit']) / current_segment_metrics['total_profit']
    
    segment_results[f'risk_{risk_group}'] = {
        'segment_size': len(segment_data),
        'current_price': current_segment_price,
        'optimal_price': optimal_price,
        'current_profit': current_segment_metrics['total_profit'],
        'optimal_profit': optimal_metrics['total_profit'],
        'improvement': improvement,
        'current_conversion': current_segment_metrics['conversion_rate'],
        'optimal_conversion': optimal_metrics['conversion_rate']
    }
    
    print(f"   {risk_group}: Current ${current_segment_price:.0f} → Optimal ${optimal_price:.0f} ({improvement:.1%} profit improvement)")

# Overall segmentation impact
print(f"\nSEGMENTATION IMPACT ANALYSIS:")
print("=" * 40)

total_current_profit = sum([result['current_profit'] for result in segment_results.values()])
total_optimal_profit = sum([result['optimal_profit'] for result in segment_results.values()])
overall_improvement = (total_optimal_profit - total_current_profit) / total_current_profit

print(f"Total current profit (all segments): ${total_current_profit:,.2f}")
print(f"Total optimal profit (all segments): ${total_optimal_profit:,.2f}")
print(f"Overall improvement from segmentation: {overall_improvement:.2%}")

# Best performing segments
best_segments = sorted(segment_results.items(), key=lambda x: x[1]['improvement'], reverse=True)[:5]
print(f"\nTOP 5 SEGMENTS BY IMPROVEMENT:")
for i, (segment, result) in enumerate(best_segments, 1):
    print(f"   {i}. {segment}: {result['improvement']:.1%} improvement (${result['current_price']:.0f} → ${result['optimal_price']:.0f})")

# Visualize segment optimization
segment_names = []
current_prices = []
optimal_prices = []
improvements = []

for segment, result in list(segment_results.items())[:8]:  # Show first 8 segments
    segment_names.append(segment.replace('_', ' ').title())
    current_prices.append(result['current_price'])
    optimal_prices.append(result['optimal_price'])
    improvements.append(result['improvement'] * 100)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Price comparison
x = np.arange(len(segment_names))
width = 0.35

axes[0].bar(x - width/2, current_prices, width, label='Current Price', color='lightblue')
axes[0].bar(x + width/2, optimal_prices, width, label='Optimal Price', color='lightcoral')
axes[0].set_xlabel('Customer Segment')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price Optimization by Segment')
axes[0].set_xticks(x)
axes[0].set_xticklabels(segment_names, rotation=45, ha='right')
axes[0].legend()

# Improvement percentages
colors = ['green' if imp > 0 else 'red' for imp in improvements]
axes[1].bar(x, improvements, color=colors, alpha=0.7)
axes[1].set_xlabel('Customer Segment')
axes[1].set_ylabel('Profit Improvement (%)')
axes[1].set_title('Profit Improvement by Segment')
axes[1].set_xticks(x)
axes[1].set_xticklabels(segment_names, rotation=45, ha='right')
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n" + "="*50)
print("Ready to proceed to Dynamic Pricing (Notebook 4)")
print("="*50)